In [17]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

ROUTER_MODEL = "llama-3.1-8b-instant"     # Super fast, low cost
EXPERT_MODEL = "llama-3.3-70b-versatile"  # High intelligence for experts

In [18]:
EXPERTS = {
    "technical": {
        "system_prompt": "You are a Senior Software Engineer. Provide concise, code-heavy solutions. Use Markdown for code blocks and explain the 'why' behind the fix.",
        "temperature": 0.3
    },
    "billing": {
        "system_prompt": "You are a Customer Billing Specialist. Be empathetic, professional, and focus on company policy (e.g., 30-day refund window). Avoid technical jargon.",
        "temperature": 0.5
    },
    "general": {
        "system_prompt": "You are a friendly, helpful general assistant. Keep answers brief and conversational.",
        "temperature": 0.7
    },
    "crypto_tool": {
        "system_prompt": "You are a Financial Data Bot. You specialize in real-time cryptocurrency price analysis.",
        "temperature": 0.1
    }
}

In [19]:
def route_prompt(user_input):
    """Classifies the user input into a specific category."""
    
    routing_instructions = f"""
    Analyze the user input and classify it into EXACTLY ONE of these categories:
    [technical, billing, general, crypto_tool].
    
    - technical: Bug reports, code errors, API issues.
    - billing: Refunds, overcharges, subscription status.
    - crypto_tool: Specific requests for the current price of Bitcoin or Ethereum.
    - general: Greetings, casual chat, or anything else.
    
    Return ONLY the category name. No punctuation, no explanation.
    """

    response = client.chat.completions.create(
        model=ROUTER_MODEL,
        messages=[
            {"role": "system", "content": routing_instructions},
            {"role": "user", "content": user_input}
        ],
        temperature=0 # Absolute consistency
    )
    
    return response.choices[0].message.content.strip().lower()

In [20]:
def get_crypto_price(coin):
    """Mock tool to fetch crypto prices."""
    mock_data = {"bitcoin": "$68,500", "ethereum": "$3,400"}
    price = mock_data.get(coin.lower(), "unknown")
    return f"The current mock price of {coin} is {price}."

In [ ]:
def process_request(user_input):
    # 1. Route the input
    category = route_prompt(user_input)
    print(f"--- [Router Log]: Routed to '{category}' ---")

    # 2. Handle the Bonus "Tool Use" Logic
    if category == "crypto_tool":
        if "bitcoin" in user_input.lower():
            return get_crypto_price("bitcoin")
        elif "ethereum" in user_input.lower():
            return get_crypto_price("ethereum")
        else:
            return "I can only check prices for Bitcoin and Ethereum right now."

    # 3. Select Expert Configuration
    config = EXPERTS.get(category, EXPERTS["general"])

    # 4. Call the Expert LLM
    response = client.chat.completions.create(
        model=EXPERT_MODEL,
        messages=[
            {"role": "system", "content": config["system_prompt"]},
            {"role": "user", "content": user_input}
        ],
        temperature=config["temperature"]
    )

    return response.choices[0].message.content

# --- TEST SUITE ---
if __name__ == "__main__":
    queries = [
        "My python script is throwing an IndexError on line 5.",
        "I was charged twice for my subscription this month.",
        "What is the current price of Bitcoin?",
        "Hey, how's your day going?"
    ]

    for q in queries:
        print(f"\nUser: {q}")
        print(f"Assistant: {process_request(q)}")
        print("-" * 30)


User: My python script is throwing an IndexError on line 5.
--- [Router Log]: Routed to 'technical' ---
Assistant: ### Debugging IndexError
To solve the `IndexError`, we need to ensure that the index we're trying to access exists in the list. 

Here's a step-by-step approach:
1. **Check the list length**: Verify that the list has enough elements to access the index at line 5.
2. **Validate the index**: Confirm that the index is within the bounds of the list (i.e., `0 <= index < len(list)`).

Let's assume your code looks something like this:
```python
my_list = [1, 2, 3]
index = 5
print(my_list[index])  # This will throw an IndexError
```
To fix this, we can add a simple check:
```python
my_list = [1, 2, 3]
index = 5

if index < len(my_list):
    print(my_list[index])
else:
    print("Index out of range")
```
Alternatively, you can use a `try-except` block to catch the `IndexError` exception:
```python
my_list = [1, 2, 3]
index = 5

try:
    print(my_list[index])
except IndexError:
   

# Unit 2 Assignment: MoE Router Observations

## Routing Accuracy
Across all test cases, the **Llama-3.1-8b-instant** router achieved 100% accuracy. By using `temperature=0`, the router consistently returned the exact category string without conversational filler, which is critical for the orchestrator's logic to function.

## Expert Persona Analysis

### 1. Technical Expert
- **Observation:** Successfully identified technical keywords ("Python", "IndexError").
- **Behavior:** The response was structured and logical. It didn't just provide a solution but explained the underlying cause and offered two distinct coding patterns (Input Validation and Exception Handling), fulfilling the "Senior Engineer" persona.

### 2. Billing Expert
- **Observation:** Correctly triggered by financial intent ("charged twice", "subscription").
- **Behavior:** This expert displayed the highest shift in "voice." It transitioned from the cold, logic-based tone of the Technical Expert to an empathetic, customer-first tone. It strictly adhered to the business constraint of the "30-day refund window."

### 3. Tool Use (Bonus)
- **Observation:** The router correctly identified the request for real-time data (`crypto_tool`).
- **Behavior:** Instead of the LLM "hallucinating" a price, the orchestrator intercepted the request and routed it to a deterministic Python function. This demonstrates a hybrid MoE architecture where "Experts" can be either LLMs or hard-coded tools.

### 4. General Expert
- **Observation:** Acted as an effective catch-all for non-specific queries ("How's your day?").
- **Behavior:** Maintained a brief and friendly tone, preventing the system from over-analyzing casual social interactions.

---

## Architectural Benefits
* **Latency Optimization:** Using a smaller, faster model for the Router minimizes the overhead of the classification step.
* **Specialization:** By separating system prompts, we can fine-tune the "creativity" (temperature) for each expert individually—keeping technical answers precise ($0.2$) and general answers fluid ($0.7$).
* **Scalability:** The system is modular; new experts or specialized tools can be added to the `MODEL_CONFIG` without rewriting the core orchestration logic.